# 02 — Data Cleaning & Feature Engineering

**Rapido Ride Analysis** | Section A, Group 1

**Objective:** Clean the raw dataset and engineer features that support our hypothesis:
> *Areas with high ride demand tend to experience higher cancellation rates due to insufficient driver availability, especially during peak hours.*

## 1. Setup & Load

In [108]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [109]:
# Load raw dataset
df = pd.read_csv('../data/raw/rapido_rides.csv')
print(f'Raw dataset: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

Raw dataset: 51000 rows × 13 columns


,services,date,time,ride_status,source,destination,duration,ride_id,distance,ride_charge,misc_charge,total_fare,payment_method
0,cab economy,2024-07-15,08:30:40.542646,completed,Balagere Harbor,Harohalli Nagar,38.0,RD3161218751875354,9.22,307.71,9.44,317.15,Amazon Pay
1,auto,2024-07-05,23:36:51.542646,completed,Basavanagudi 3rd Block,Bikasipura 1st Stage,14.0,RD8171514284594096,4.29,131.22,1.96,133.18,Paytm
2,auto,2024-07-23,11:05:37.542646,completed,Kormangala,Kothaguda Terrace,22.0,RD9376481122237926,7.35,NaN,NaN,NaN,NaN
3,cab economy,2024-06-24,08:45:10.542646,completed,koramangala,Kanakapura Arc,84.0,RD3676889143182765,17.48,558.48,1.57,560.05,QR scan
4,cab economy,2024-07-15,00:26:44.542646,completed,Ganganagar Cove,Basaveshwaranagar Colony,38.0,RD6639410275948084,17.63,543.02,9.17,552.19,Amazon Pay


In [110]:
# Initial inspection
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Missing Values ===')
print(df.isnull().sum())
print(f'\n=== Describe ===')
df.describe()

=== Data Types ===
services           object
date               object
time               object
ride_status        object
source             object
destination        object
duration          float64
ride_id            object
distance          float64
ride_charge       float64
misc_charge       float64
total_fare        float64
payment_method     object
dtype: object

=== Missing Values ===
services              0
date                  0
time                  0
ride_status           0
source                0
destination           0
duration              0
ride_id               0
distance              0
ride_charge       10346
misc_charge       10346
total_fare        10346
payment_method    12398
dtype: int64

=== Describe ===


,duration,distance,ride_charge,misc_charge,total_fare
count,51000.000000,51000.000000,40654.000000,40654.000000,40654.000000
mean,24.536157,7.080707,150.795340,7.966225,158.761565
std,19.086564,5.530099,144.497965,7.888836,144.716308
min,5.000000,0.800000,15.550000,0.000000,17.040000
25%,12.000000,3.310000,57.140000,2.290000,65.462500
50%,19.000000,5.500000,99.850000,5.520000,108.510000
75%,31.000000,9.070000,192.960000,11.070000,201.010000
max,180.000000,45.000000,1963.360000,50.000000,1990.940000


## 2. Duplicate Removal

In [111]:
# Check for duplicate ride_ids
total = len(df)
unique = df['ride_id'].nunique()
dupes = total - unique
print(f'Total rows: {total}')
print(f'Unique ride_ids: {unique}')
print(f'Duplicate ride_ids: {dupes}')

# Drop duplicates, keep first occurrence
df = df.drop_duplicates(subset='ride_id', keep='first')
print(f'\nAfter dedup: {len(df)} rows (removed {dupes})')

Total rows: 51000
Unique ride_ids: 50000
Duplicate ride_ids: 1000

After dedup: 50000 rows (removed 1000)


## 3. Service Name Standardization
The `services` column has variants due to casing and whitespace. Standardizing to 5 clean categories.

In [112]:
# Before: show all variants
print('BEFORE — Service variants:')
print(df['services'].value_counts())
print(f'\nUnique values: {df["services"].nunique()}')

BEFORE — Service variants:
services
bike            16658
auto            10923
cab economy      8972
parcel           6634
bike lite        4309
bike              239
 bike             210
Bike              202
BIKE              202
auto              155
AUTO              152
 auto             138
Auto              138
cab economy       134
 cab economy      127
CAB ECONOMY       127
Cab Economy       120
Parcel             91
parcel             82
PARCEL             79
 parcel            76
BIKE LITE          74
 bike lite         56
Bike Lite          56
bike lite          46
Name: count, dtype: int64

Unique values: 25


In [113]:
# Standardize: strip whitespace, lowercase, replace spaces with underscores
df['services'] = df['services'].str.strip().str.lower().str.replace(' ', '_')

print('AFTER — Service categories:')
print(df['services'].value_counts())
print(f'\nUnique values: {df["services"].nunique()}')

AFTER — Service categories:
services
bike           17511
auto           11506
cab_economy     9480
parcel          6962
bike_lite       4541
Name: count, dtype: int64

Unique values: 5


## 4. Date & Time Parsing
The `date` column has mixed formats: `YYYY-MM-DD` and `DD/MM/YYYY`. We need to handle both.

In [114]:
# Check invalid dates
test_parse = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')
invalid_count = test_parse.isna().sum()
print(f'Dates that fail YYYY-MM-DD parsing: {invalid_count}')
print(f'Sample invalid dates: {df.loc[test_parse.isna(), "date"].head(5).tolist()}')

Dates that fail YYYY-MM-DD parsing: 1491
Sample invalid dates: ['12/08/2024', '04/08/2024', '23/07/2024', '16/08/2024', '02/08/2024']


In [115]:
# Parse with mixed format support (dayfirst=True for DD/MM/YYYY)
df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=True)

print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Invalid dates remaining: {df["date"].isna().sum()}')

Date range: 2024-06-17 00:00:00 to 2024-08-16 00:00:00
Invalid dates remaining: 0


In [116]:
# Create unified timestamp by combining date + time
df['time_parsed'] = pd.to_datetime(df['time'], format='mixed', errors='coerce')
df['timestamp'] = df['date'] + pd.to_timedelta(df['time_parsed'].dt.strftime('%H:%M:%S'))

# Drop helper column
df = df.drop(columns=['time_parsed'])

print(f'Timestamp range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'Invalid timestamps: {df["timestamp"].isna().sum()}')
df[['date', 'time', 'timestamp']].head()

Timestamp range: 2024-06-17 00:00:55 to 2024-08-16 23:59:24
Invalid timestamps: 0


,date,time,timestamp
0,2024-07-15,08:30:40.542646,2024-07-15 08:30:40
1,2024-07-05,23:36:51.542646,2024-07-05 23:36:51
2,2024-07-23,11:05:37.542646,2024-07-23 11:05:37
3,2024-06-24,08:45:10.542646,2024-06-24 08:45:10
4,2024-07-15,00:26:44.542646,2024-07-15 00:26:44


## 5. Missing Value Handling (Business Logic)

**Rules:**
- **Cancelled rides** → fare/charge fields = 0, payment = 'not_applicable' (they never completed)
- **Completed rides with missing fare** → impute using service-specific fare formula

In [117]:
# Current missing value state
print('Missing values BEFORE handling:')
print(df.isnull().sum())

cancelled = df['ride_status'].str.strip().str.lower() == 'cancelled'
completed = df['ride_status'].str.strip().str.lower() == 'completed'

print(f'\nCancelled rides: {cancelled.sum()}')
print(f'Completed rides: {completed.sum()}')
print(f'Completed with missing fare: {(completed & df["ride_charge"].isna()).sum()}')

Missing values BEFORE handling:
services              0
date                  0
time                  0
ride_status           0
source                0
destination           0
duration              0
ride_id               0
distance              0
ride_charge       10149
misc_charge       10149
total_fare        10149
payment_method    12155
timestamp             0
dtype: int64

Cancelled rides: 5212
Completed rides: 44788
Completed with missing fare: 4937


In [118]:
# 5a. Handle cancelled rides — set fare fields to 0
df.loc[cancelled, 'ride_charge'] = 0
df.loc[cancelled, 'misc_charge'] = 0
df.loc[cancelled, 'total_fare'] = 0
df.loc[cancelled, 'payment_method'] = 'not_applicable'

print(f'Set {cancelled.sum()} cancelled rides: fare=0, payment=not_applicable')

Set 5212 cancelled rides: fare=0, payment=not_applicable


In [119]:
# 5b. Impute missing fares for COMPLETED rides using service-specific pricing
# Rapido-style fare: base_fare + (per_km × distance) + (per_min × duration)
fare_rules = {
    'bike':        {'base': 15, 'per_km': 5,  'per_min': 1.0},
    'bike_lite':   {'base': 10, 'per_km': 4,  'per_min': 0.8},
    'auto':        {'base': 25, 'per_km': 12, 'per_min': 1.5},
    'cab_economy': {'base': 50, 'per_km': 14, 'per_min': 2.0},
    'parcel':      {'base': 20, 'per_km': 8,  'per_min': 0.5},
}

missing_fare_mask = completed & df['ride_charge'].isna()
impute_count = 0

for svc, rules in fare_rules.items():
    svc_mask = missing_fare_mask & (df['services'] == svc)
    if svc_mask.sum() > 0:
        imputed = rules['base'] + (rules['per_km'] * df.loc[svc_mask, 'distance']) + (rules['per_min'] * df.loc[svc_mask, 'duration'])
        # Add slight noise for realism
        noise = np.random.uniform(0.95, 1.05, svc_mask.sum())
        df.loc[svc_mask, 'ride_charge'] = np.round(imputed * noise, 2)
        impute_count += svc_mask.sum()

# Fill misc_charge if still missing (use median)
median_misc = df.loc[completed & df['misc_charge'].notna(), 'misc_charge'].median()
df.loc[completed & df['misc_charge'].isna(), 'misc_charge'] = round(median_misc, 2)

# Recalculate total_fare
df.loc[completed, 'total_fare'] = np.round(df.loc[completed, 'ride_charge'] + df.loc[completed, 'misc_charge'], 2)

# Fill missing payment_method with random choice from existing distribution
payment_options = df.loc[completed & df['payment_method'].notna() & (df['payment_method'] != 'not_applicable'), 'payment_method']
missing_payment = completed & (df['payment_method'].isna())
if missing_payment.sum() > 0:
    df.loc[missing_payment, 'payment_method'] = np.random.choice(payment_options.values, size=missing_payment.sum())

print(f'Imputed fare for {impute_count} completed rides')
print(f'\nMissing values AFTER handling:')
print(df.isnull().sum())

Imputed fare for 4937 completed rides

Missing values AFTER handling:
services          0
date              0
time              0
ride_status       0
source            0
destination       0
duration          0
ride_id           0
distance          0
ride_charge       0
misc_charge       0
total_fare        0
payment_method    0
timestamp         0
dtype: int64


## 6. Anomaly Filtering
Remove physically implausible rides.

In [120]:
before = len(df)

# Calculate speed for anomaly detection
df['_speed'] = df['distance'] / (df['duration'] / 60)

# Flag anomalies
anomaly_speed = df['_speed'] > 80  # > 80 km/h impossible in Bangalore traffic
anomaly_duration = df['duration'] > 120  # > 2 hours is extreme
anomaly_distance = df['distance'] > 50  # > 50 km unlikely for intra-city

total_anomalies = anomaly_speed | anomaly_duration | anomaly_distance
print(f'Anomalies found:')
print(f'  Speed > 80 km/h: {anomaly_speed.sum()}')
print(f'  Duration > 120 min: {anomaly_duration.sum()}')
print(f'  Distance > 50 km: {anomaly_distance.sum()}')
print(f'  Total (union): {total_anomalies.sum()}')

df = df[~total_anomalies].copy()
df = df.drop(columns=['_speed'])

print(f'\nRows: {before} → {len(df)} (removed {before - len(df)})')

Anomalies found:
  Speed > 80 km/h: 2
  Duration > 120 min: 197
  Distance > 50 km: 0
  Total (union): 199

Rows: 50000 → 49801 (removed 199)


## 7. Categorical Standardization

In [121]:
# Standardize all text columns
df['ride_status'] = df['ride_status'].str.strip().str.lower()
df['source'] = df['source'].str.strip().str.lower()
df['destination'] = df['destination'].str.strip().str.lower()
df['payment_method'] = df['payment_method'].str.strip().str.lower()

print('Ride status values:', df['ride_status'].unique())
print('Payment methods:', df['payment_method'].unique())
print(f'Unique sources: {df["source"].nunique()}')
print(f'Unique destinations: {df["destination"].nunique()}')

Ride status values: ['completed' 'cancelled']
Payment methods: ['amazon pay' 'paytm' 'qr scan' 'gpay' 'not_applicable']
Unique sources: 12901
Unique destinations: 12974


## 8. Feature Engineering

Derived columns designed to support our core hypothesis:
> *High-demand areas experience higher cancellation rates due to insufficient driver availability, especially during peak hours.*

### Categories of derived features:
1. **Time-based** — for demand/cancellation pattern analysis
2. **Demand & cancellation metrics** — zone-level demand and cancellation rates
3. **Ride economics** — fare and speed metrics
4. **Flags & categories** — for segmentation and filtering

### 8.1 Time-Based Features

In [122]:
# Extract time components from timestamp
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday

# Weekend flag
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Peak hour flag (8-10 AM morning rush, 5-7 PM evening rush)
df['peak_hour'] = df['hour'].isin([8, 9, 10, 17, 18, 19]).astype(int)

# Time slot — granular bucketing for demand curves
def get_time_slot(hour):
    if 6 <= hour < 10:
        return 'morning_rush'
    elif 10 <= hour < 13:
        return 'late_morning'
    elif 13 <= hour < 16:
        return 'afternoon'
    elif 16 <= hour < 20:
        return 'evening_rush'
    elif 20 <= hour < 23:
        return 'night'
    else:
        return 'late_night'

df['time_slot'] = df['hour'].apply(get_time_slot)

# Drop original time column (replaced by timestamp)
df = df.drop(columns=['time'])

print('Time features created:')
print(f'  Peak hour rides: {df["peak_hour"].sum()} ({df["peak_hour"].mean()*100:.1f}%)')
print(f'  Weekend rides: {df["is_weekend"].sum()} ({df["is_weekend"].mean()*100:.1f}%)')
print(f'\nTime slot distribution:')
print(df['time_slot'].value_counts())

Time features created:
  Peak hour rides: 12503 (25.1%)
  Weekend rides: 13075 (26.3%)

Time slot distribution:
time_slot
late_night      14387
morning_rush     8406
evening_rush     8221
late_morning     6330
night            6251
afternoon        6206
Name: count, dtype: int64


### 8.2 Demand & Cancellation Zone Features
These directly test our hypothesis — do high-demand zones have higher cancellation rates?

In [123]:
# Binary completion flag
df['completed'] = (df['ride_status'] == 'completed').astype(int)

source_stats = df.groupby('source').agg(
    _src_count=('ride_id', 'count'),
    source_cancel_rate=('completed', lambda x: round(1 - x.mean(), 4))
)
df = df.merge(source_stats, on='source', how='left')

# Demand zone classification (high/medium/low based on ride count)
demand_q33 = df['_src_count'].quantile(0.33)
demand_q66 = df['_src_count'].quantile(0.66)

def classify_demand(count):
    if count >= demand_q66:
        return 'high_demand'
    elif count >= demand_q33:
        return 'medium_demand'
    else:
        return 'low_demand'

df['demand_zone'] = df['_src_count'].apply(classify_demand)
df = df.drop(columns=['_src_count'])  # drop intermediate count

print('Demand zone distribution:')
print(df['demand_zone'].value_counts())
print(f'\nCancel rate by demand zone (testing our hypothesis):')
print(df.groupby('demand_zone')['source_cancel_rate'].mean().round(4))

Demand zone distribution:
demand_zone
medium_demand    17590
high_demand      17354
low_demand       14857
Name: count, dtype: int64

Cancel rate by demand zone (testing our hypothesis):
demand_zone
high_demand      0.1039
low_demand       0.1065
medium_demand    0.1020
Name: source_cancel_rate, dtype: float64


In [124]:
# ─── DEMAND INTENSITY ───
# Normalized rides per hour (1.0 = average) to identify demand spikes
hourly_demand = df.groupby('hour')['ride_id'].count()
avg_hourly = hourly_demand.mean()
demand_intensity_map = round(hourly_demand / avg_hourly, 2)
df['demand_intensity'] = df['hour'].map(demand_intensity_map)

print('Demand intensity by hour (1.0 = average):')
print(demand_intensity_map.to_string())

Demand intensity by hour (1.0 = average):
hour
0     0.98
1     1.01
2     0.96
3     0.99
4     1.00
5     1.01
6     1.00
7     1.02
8     1.00
9     1.03
10    1.00
11    1.05
12    1.00
13    1.02
14    0.97
15    1.00
16    0.98
17    1.00
18    1.00
19    0.99
20    1.00
21    1.00
22    1.01
23    0.96


In [125]:
# ─── SERVICE-LEVEL CANCELLATION RATE ───
# Which services get cancelled more?
service_cancel = df.groupby('services')['completed'].apply(lambda x: round(1 - x.mean(), 4)).rename('service_cancel_rate')
df = df.merge(service_cancel, on='services', how='left')

print('Cancellation rate by service:')
print(df.groupby('services')['service_cancel_rate'].first().sort_values(ascending=False))

Cancellation rate by service:
services
cab_economy    0.1074
parcel         0.1053
bike           0.1036
auto           0.1028
bike_lite      0.0995
Name: service_cancel_rate, dtype: float64


### 8.3 Ride Economics Features

In [126]:
# Average speed (km/h) — indicates traffic conditions
df['avg_speed_kmh'] = round(df['distance'] / (df['duration'] / 60), 2)

# Fare per km — pricing analysis
df['fare_per_km'] = round(df['total_fare'] / df['distance'], 2)
# Handle division by zero / cancelled rides
df.loc[df['distance'] == 0, 'fare_per_km'] = 0
df.loc[df['ride_status'] == 'cancelled', 'fare_per_km'] = 0

print('Ride economics summary (completed rides only):')
completed_df = df[df['ride_status'] == 'completed']
for col in ['avg_speed_kmh', 'fare_per_km']:
    print(f'  {col}: mean={completed_df[col].mean():.2f}, median={completed_df[col].median():.2f}')

Ride economics summary (completed rides only):
  avg_speed_kmh: mean=18.37, median=17.76
  fare_per_km: mean=23.37, median=21.34


### 8.4 Ride Categories & Flags

In [127]:
# Distance category — short/medium/long rides
def categorize_distance(d):
    if d <= 5:
        return 'short'
    elif d <= 15:
        return 'medium'
    else:
        return 'long'

df['distance_category'] = df['distance'].apply(categorize_distance)

# Duration category
def categorize_duration(d):
    if d <= 15:
        return 'quick'
    elif d <= 30:
        return 'moderate'
    else:
        return 'long'

df['duration_category'] = df['duration'].apply(categorize_duration)

# Fare category — budget/standard/premium
completed_df = df[df['ride_status'] == 'completed']
fare_q33 = completed_df['total_fare'].quantile(0.33)
fare_q66 = completed_df['total_fare'].quantile(0.66)

def categorize_fare(f):
    if f == 0:
        return 'cancelled'
    elif f <= fare_q33:
        return 'budget'
    elif f <= fare_q66:
        return 'standard'
    else:
        return 'premium'

df['fare_category'] = df['total_fare'].apply(categorize_fare)

print('Distance categories:')
print(df['distance_category'].value_counts())
print(f'\nDuration categories:')
print(df['duration_category'].value_counts())
print(f'\nFare categories:')
print(df['fare_category'].value_counts())

Distance categories:
distance_category
medium    23294
short     22535
long       3972
Name: count, dtype: int64

Duration categories:
duration_category
moderate    18659
quick       18651
long        12491
Name: count, dtype: int64

Fare categories:
fare_category
premium      15171
standard     14725
budget       14725
cancelled     5180
Name: count, dtype: int64


## 9. Data Type Optimization

In [128]:
# Convert to efficient types
df['duration'] = df['duration'].astype(int)
df['services'] = df['services'].astype('category')
df['ride_status'] = df['ride_status'].astype('category')
df['payment_method'] = df['payment_method'].astype('category')
df['time_slot'] = df['time_slot'].astype('category')
df['demand_zone'] = df['demand_zone'].astype('category')
df['distance_category'] = df['distance_category'].astype('category')
df['duration_category'] = df['duration_category'].astype('category')
df['fare_category'] = df['fare_category'].astype('category')

print('Optimized dtypes:')
print(df.dtypes)

Optimized dtypes:
services                     category
date                   datetime64[ns]
ride_status                  category
source                         object
destination                    object
duration                        int64
ride_id                        object
distance                      float64
ride_charge                   float64
misc_charge                   float64
total_fare                    float64
payment_method               category
timestamp              datetime64[ns]
year                            int32
month                           int32
hour                            int32
day_of_week                     int32
is_weekend                      int64
peak_hour                       int64
time_slot                    category
completed                       int64
source_cancel_rate            float64
demand_zone                  category
demand_intensity              float64
service_cancel_rate           float64
avg_speed_kmh                 fl

## 10. Final Validation

In [129]:
print(f'=== FINAL DATASET ===')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values ✓')
print(f'\n=== Duplicates ===')
print(f'Duplicate ride_ids: {df.duplicated(subset="ride_id").sum()} ✓')
print(f'\n=== Column List ===')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col} ({df[col].dtype})')

=== FINAL DATASET ===
Shape: 49801 rows × 30 columns

=== Missing Values ===
No missing values ✓

=== Duplicates ===
Duplicate ride_ids: 0 ✓

=== Column List ===
   1. services (category)
   2. date (datetime64[ns])
   3. ride_status (category)
   4. source (object)
   5. destination (object)
   6. duration (int64)
   7. ride_id (object)
   8. distance (float64)
   9. ride_charge (float64)
  10. misc_charge (float64)
  11. total_fare (float64)
  12. payment_method (category)
  13. timestamp (datetime64[ns])
  14. year (int32)
  15. month (int32)
  16. hour (int32)
  17. day_of_week (int32)
  18. is_weekend (int64)
  19. peak_hour (int64)
  20. time_slot (category)
  21. completed (int64)
  22. source_cancel_rate (float64)
  23. demand_zone (category)
  24. demand_intensity (float64)
  25. service_cancel_rate (float64)
  26. avg_speed_kmh (float64)
  27. fare_per_km (float64)
  28. distance_category (category)
  29. duration_category (category)
  30. fare_category (category)


In [130]:
# Summary statistics of the cleaned dataset
print('=== Numeric Summary ===')
df.describe().round(2)

=== Numeric Summary ===


,date,duration,distance,ride_charge,misc_charge,total_fare,timestamp,year,month,hour,day_of_week,is_weekend,peak_hour,completed,source_cancel_rate,demand_intensity,service_cancel_rate,avg_speed_kmh,fare_per_km
count,49801,49801.00,49801.00,49801.00,49801.00,49801.00,49801,49801.0,49801.00,49801.00,49801.00,49801.00,49801.00,49801.00,49801.00,49801.00,49801.00,49801.00,49801.00
mean,2024-07-17 03:20:59.324110080,24.09,6.99,130.41,6.90,137.31,2024-07-17 15:20:03.586173184,2024.0,7.04,11.49,2.92,0.26,0.25,0.90,0.10,1.00,0.10,18.31,20.94
min,2024-06-17 00:00:00,5.00,0.80,0.00,0.00,0.00,2024-06-17 00:00:55,2024.0,6.00,0.00,0.00,0.00,0.00,0.00,0.00,0.96,0.10,2.07,0.00
25%,2024-07-02 00:00:00,12.00,3.31,45.83,1.63,53.09,2024-07-02 12:10:00,2024.0,7.00,6.00,1.00,0.00,0.00,1.00,0.00,0.99,0.10,13.26,12.86
50%,2024-07-17 00:00:00,19.00,5.48,85.95,5.46,93.84,2024-07-17 15:00:47,2024.0,7.00,11.00,3.00,0.00,0.00,1.00,0.00,1.00,0.10,17.69,19.85
75%,2024-08-01 00:00:00,31.00,9.02,174.47,9.23,182.17,2024-08-01 19:29:48,2024.0,8.00,17.00,5.00,1.00,1.00,1.00,0.20,1.01,0.11,22.30,27.96
max,2024-08-16 00:00:00,120.00,45.00,1291.91,50.00,1301.58,2024-08-16 23:59:24,2024.0,8.00,23.00,6.00,1.00,1.00,1.00,1.00,1.05,0.11,73.10,110.18
std,NaN,17.57,5.32,133.65,7.46,134.66,NaN,0.0,0.70,6.90,1.98,0.44,0.43,0.31,0.15,0.02,0.00,6.85,12.63


In [131]:
# Key hypothesis validation preview
print('=== HYPOTHESIS CHECK: High Demand + Peak Hour → Higher Cancellations? ===')
print()
pivot = df.groupby(['demand_zone', 'peak_hour']).agg(
    rides=('ride_id', 'count'),
    cancellation_rate=('completed', lambda x: round((1 - x.mean()) * 100, 2))
).reset_index()
pivot['peak_hour'] = pivot['peak_hour'].map({0: 'Off-Peak', 1: 'Peak'})
print(pivot.to_string(index=False))

=== HYPOTHESIS CHECK: High Demand + Peak Hour → Higher Cancellations? ===

  demand_zone peak_hour  rides  cancellation_rate
  high_demand  Off-Peak  13016               9.56
  high_demand      Peak   4338              12.89
   low_demand  Off-Peak  11105               9.57
   low_demand      Peak   3752              13.86
medium_demand  Off-Peak  13177               9.26
medium_demand      Peak   4413              13.01


## 11. Export Cleaned Dataset

In [132]:
# Save to processed directory
output_path = '../data/processed/cleaned_rides.csv'
df.to_csv(output_path, index=False)

print(f'Cleaned dataset saved to: {output_path}')
print(f'Final shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nOriginal columns (cleaned):')
original_cols = ['services', 'date', 'ride_status', 'source', 'destination',
                 'duration', 'ride_id', 'distance', 'ride_charge', 'misc_charge',
                 'total_fare', 'payment_method']
for c in original_cols:
    if c in df.columns: print(f'  ● {c}')
print(f'\nDerived columns ({len([c for c in df.columns if c not in original_cols])} new):')
for c in df.columns:
    if c not in original_cols: print(f'  + {c}')

Cleaned dataset saved to: ../data/processed/cleaned_rides.csv
Final shape: 49801 rows × 30 columns

Original columns (cleaned):
  ● services
  ● date
  ● ride_status
  ● source
  ● destination
  ● duration
  ● ride_id
  ● distance
  ● ride_charge
  ● misc_charge
  ● total_fare
  ● payment_method

Derived columns (18 new):
  + timestamp
  + year
  + month
  + hour
  + day_of_week
  + is_weekend
  + peak_hour
  + time_slot
  + completed
  + source_cancel_rate
  + demand_zone
  + demand_intensity
  + service_cancel_rate
  + avg_speed_kmh
  + fare_per_km
  + distance_category
  + duration_category
  + fare_category
